# VerSe'20（OSF + osfclient）→ Google Drive

在 Colab 中挂载 Google Drive，使用 **`osfclient`** 从 OSF 项目 **[b2wxj](https://osf.io/b2wxj/overview)** 将数据下载到 `dataset/pretrain/VerSe/download/`，再解压到同级的 `unzip/`。

- 项目页：<https://osf.io/b2wxj/overview>；项目 ID：`b2wxj`。
- **公开项目**一般无需登录；**私有**请在 Colab 中设置环境变量 **`OSF_TOKEN`**（OSF 个人访问令牌），或按 [osfclient 文档](https://osfclient.readthedocs.io/) 配置 `OSF_USERNAME` / `OSF_PASSWORD`。

运行时会自动创建（若不存在）：`VerSe/download`、`VerSe/unzip`、`VerSe/npy`（`npy` 供后续预处理占位）。

**备选**：若 boneScreen S3 直链可用，仍可用 `wget`/`curl` 自行下载三个 `dataset-verse20*.zip` 到 `download/`；或 **Academic Torrents**：[VerSe'20 torrent](http://academictorrents.com/details/0ac07fd4ddf1802208f88c61c5ccf7d029d87a18)（约 38GB）。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os

BASE = "/content/drive/MyDrive/dataset/pretrain/VerSe"
DEST = os.path.join(BASE, "download")
UNZIP_ROOT = os.path.join(BASE, "unzip")
NPY_ROOT = os.path.join(BASE, "npy")

for p in (BASE, DEST, UNZIP_ROOT, NPY_ROOT):
    os.makedirs(p, exist_ok=True)

print("已创建/确认目录:")
print("  ", BASE)
print("  ", DEST)
print("  ", UNZIP_ROOT)
print("  ", NPY_ROOT)

In [ ]:
!pip install -q osfclient

import os
import shutil
import subprocess

DEST = "/content/drive/MyDrive/dataset/pretrain/VerSe/download"
OSF_PROJECT_ID = "b2wxj"
os.makedirs(DEST, exist_ok=True)

cfg_path = os.path.join(DEST, ".osfcli.config")
with open(cfg_path, "w", encoding="utf-8") as f:
    f.write("[osf]\n")
    f.write(f"project = {OSF_PROJECT_ID}\n")

osf_exe = shutil.which("osf")
if not osf_exe:
    raise RuntimeError("未找到 osf 命令，请确认 pip install osfclient 已成功")

print(f"OSF: https://osf.io/{OSF_PROJECT_ID}/overview")
print(f"已写入 {cfg_path}")
print(f"下载目录: {DEST}")
print("执行 osf clone ...")

r = subprocess.run([osf_exe, "clone"], cwd=DEST, capture_output=True, text=True)
if r.returncode != 0:
    r2 = subprocess.run(
        [osf_exe, "-p", OSF_PROJECT_ID, "clone", "."],
        cwd=DEST,
        capture_output=True,
        text=True,
    )
    if r2.returncode != 0:
        print("=== 第一次 osf clone (cwd=download) stderr ===")
        print((r.stderr or "")[-4000:])
        print("=== 第二次 osf -p <id> clone . stderr ===")
        print((r2.stderr or "")[-4000:])
        print("若项目为私有，请在 Colab 设置 OSF_TOKEN 或 OSF_USERNAME/OSF_PASSWORD 后重试。")
        raise RuntimeError("osfclient 下载失败，见上方输出")

print("\n完成。download 下顶层条目（节选）:")
names = sorted(os.listdir(DEST))
print(names[:50] if len(names) > 50 else names)
print(f"共 {len(names)} 项（含隐藏配置等）")

In [ ]:
# 解压：递归查找 download 下所有 .zip（OSF/osf clone 可能在子目录）；解压到 unzip/<相对路径去扩展名>/
import os
import zipfile

DOWNLOAD = "/content/drive/MyDrive/dataset/pretrain/VerSe/download"
UNZIP_ROOT = "/content/drive/MyDrive/dataset/pretrain/VerSe/unzip"
os.makedirs(UNZIP_ROOT, exist_ok=True)

zip_paths = []
for root, _, files in os.walk(DOWNLOAD):
    if "__MACOSX" in root.replace("\\", "/"):
        continue
    for f in files:
        if f.lower().endswith(".zip"):
            zip_paths.append(os.path.join(root, f))
zip_paths.sort()
failed = []

if not zip_paths:
    print(f"未在 {DOWNLOAD} 下发现 .zip（若已解压并删除压缩包可忽略本格）")
else:
    for i, zpath in enumerate(zip_paths, start=1):
        rel = os.path.relpath(zpath, DOWNLOAD)
        rel_no_ext = os.path.splitext(rel)[0]
        out_dir = os.path.join(UNZIP_ROOT, rel_no_ext)
        os.makedirs(out_dir, exist_ok=True)
        print(f"\n[{i}/{len(zip_paths)}] {rel}")
        print(f"    -> {out_dir}")
        try:
            with zipfile.ZipFile(zpath, "r") as zf:
                bad = zf.testzip()
                if bad is not None:
                    raise zipfile.BadZipFile(f"损坏条目: {bad}")
                zf.extractall(out_dir)
            print("    OK")
        except Exception as e:
            failed.append((rel, str(e)))
            print(f"    FAILED: {e}")

    print("\n" + "=" * 60)
    if failed:
        print(f"解压失败 {len(failed)} 个：")
        for zname, err in failed:
            print(f"  - {zname}: {err}")
        raise RuntimeError("部分 zip 解压失败，见上方列表")
    print("全部 zip 解压完成。")
    print("unzip 下顶层：", sorted(os.listdir(UNZIP_ROOT)))

In [ ]:
import os
from collections import Counter, defaultdict

UNZIP_ROOT = "/content/drive/MyDrive/dataset/pretrain/VerSe/unzip"

MAX_SAMPLES_PER_EXT = 5
MAX_TREE_DEPTH = 4
MAX_ITEMS_PER_DIR = 25


def norm_rel(path, root):
    return os.path.relpath(path, root).replace("\\", "/")


def walk_summary(root):
    ext_counter = Counter()
    dir_file_counter = {}
    samples_by_ext = defaultdict(list)
    nii_like = []
    nii_gz = []
    json_files = []
    csv_files = []
    txt_files = []

    all_dirs = 0
    all_files = 0

    for dirpath, dirnames, filenames in os.walk(root):
        all_dirs += len(dirnames)
        all_files += len(filenames)
        rel_dir = norm_rel(dirpath, root)
        dir_file_counter[rel_dir] = len(filenames)

        for fn in filenames:
            full = os.path.join(dirpath, fn)
            rel = norm_rel(full, root)

            if fn.lower().endswith(".nii.gz"):
                ext = ".nii.gz"
            else:
                ext = os.path.splitext(fn)[1].lower() or "(no_ext)"

            ext_counter[ext] += 1

            if len(samples_by_ext[ext]) < MAX_SAMPLES_PER_EXT:
                samples_by_ext[ext].append(rel)

            low = fn.lower()
            if low.endswith(".nii.gz") or low.endswith(".nii"):
                nii_like.append(rel)
            if low.endswith(".nii.gz"):
                nii_gz.append(rel)
            if low.endswith(".json"):
                json_files.append(rel)
            if low.endswith(".csv"):
                csv_files.append(rel)
            if low.endswith(".txt"):
                txt_files.append(rel)

    return {
        "ext_counter": ext_counter,
        "samples_by_ext": samples_by_ext,
        "dir_file_counter": dir_file_counter,
        "all_dirs": all_dirs,
        "all_files": all_files,
        "nii_like": nii_like,
        "nii_gz": nii_gz,
        "json_files": json_files,
        "csv_files": csv_files,
        "txt_files": txt_files,
    }


def print_tree(path, root, prefix="", depth=0, max_depth=4, max_items=25):
    name = os.path.basename(path.rstrip("/")) or path
    if depth == 0:
        print(f"{name}/")

    if depth >= max_depth:
        print(f"{prefix}... (depth limit)")
        return

    try:
        entries = sorted(os.listdir(path))
    except Exception as e:
        print(f"{prefix}[无法访问] {e}")
        return

    shown = 0
    for entry in entries:
        if shown >= max_items:
            print(f"{prefix}... (remaining {len(entries) - shown} items)")
            break

        full = os.path.join(path, entry)
        if os.path.isdir(full):
            print(f"{prefix}{entry}/")
            print_tree(
                full,
                root,
                prefix=prefix + "    ",
                depth=depth + 1,
                max_depth=max_depth,
                max_items=max_items,
            )
        else:
            tag = ""
            low = entry.lower()
            if low.endswith(".nii.gz"):
                tag = " [nii.gz]"
            elif low.endswith(".json"):
                tag = " [json]"
            elif low.endswith(".csv"):
                tag = " [csv]"
            print(f"{prefix}{entry}{tag}")
        shown += 1


if not os.path.isdir(UNZIP_ROOT):
    print(f"目录不存在: {UNZIP_ROOT}")
else:
    info = walk_summary(UNZIP_ROOT)

    print("=" * 80)
    print("VerSe unzip 目录概览")
    print("根路径:", UNZIP_ROOT)
    print("=" * 80)

    print("\n[1] 总体规模")
    print(f"目录总数: {info['all_dirs']}")
    print(f"文件总数: {info['all_files']}")
    print(f".nii/.nii.gz 总数: {len(info['nii_like'])}")
    print(f"其中 .nii.gz 总数: {len(info['nii_gz'])}")
    print(f".json 总数: {len(info['json_files'])}")
    print(f".csv 总数: {len(info['csv_files'])}")
    print(f".txt 总数: {len(info['txt_files'])}")

    print("\n[2] 扩展名统计（前 15 个）")
    for ext, cnt in info["ext_counter"].most_common(15):
        print(f"{ext:12s} {cnt}")

    print("\n[3] 各扩展名样例路径")
    for ext, paths in sorted(info["samples_by_ext"].items(), key=lambda x: (-info["ext_counter"][x[0]], x[0])):
        print(f"\n{ext} ({info['ext_counter'][ext]}):")
        for p in paths:
            print(f"  - {p}")

    print("\n[4] 文件最多的目录（前 15 个）")
    dense_dirs = sorted(info["dir_file_counter"].items(), key=lambda x: (-x[1], x[0]))[:15]
    for d, n in dense_dirs:
        print(f"{d}: {n}")

    print("\n[5] .nii.gz 样例（前 20 个）")
    for p in info["nii_gz"][:20]:
        print("  -", p)

    print("\n[6] 顶层截断树")
    print_tree(UNZIP_ROOT, UNZIP_ROOT, max_depth=MAX_TREE_DEPTH, max_items=MAX_ITEMS_PER_DIR)

    print("\n完成。预处理输出可写到同级的 `npy/`。")